In [3]:
import numpy as np
import pandas as pd
def warn(*args, **kwargs): pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')

In [4]:
URL = ("https://web.archive.org/web/20230902185326/"
 "https://en.wikipedia.org/wiki/List_of_countries_by_GDP_%28nominal%29")
table = pd.read_html(URL)
print(f"Jumlah tabel: {len(table)}")
for i, t in enumerate(table):
 print(f"table[{i}] -> shape: {t.shape}")

Jumlah tabel: 8
table[0] -> shape: (3, 3)
table[1] -> shape: (1, 1)
table[2] -> shape: (1, 3)
table[3] -> shape: (214, 8)
table[4] -> shape: (9, 2)
table[5] -> shape: (7, 2)
table[6] -> shape: (13, 2)
table[7] -> shape: (2, 2)


In [9]:
df = table[3]
print(df.shape)
print(df.head())

(214, 8)
               0         1          2          3          4          5  \
0          World         —  105568776       2023  100562011       2022   
1  United States  Americas   26854599       2023   25462700       2022   
2          China      Asia   19373586  [n 1]2023   17963171  [n 3]2022   
3          Japan      Asia    4409738       2023    4231141       2022   
4        Germany    Europe    4308854       2023    4072192       2022   

          6          7  
0  96698005       2021  
1  23315081       2021  
2  17734131  [n 1]2021  
3   4940878       2021  
4   4259935       2021  


Ambil tabel utama & rapikan menjadi Top 10

Mengambil tabel utama, meratakan header bertingkat, lalu menyaring 10 ekonomi terbesar

In [12]:
df = table[3]
df.columns = range(df.shape[1]) # ratakan header bertingkat
df1 = df[[0, 1, 2, 3]] # kolom: negara, region, nilai & tahun (IMF)
df1 = df1.iloc[0:11] # ambil Top 10 (+ baris 'World')
df1.columns = ['Country', 'UN Region', 'IMF Forecast', 'IMF Year']
df1

,Country,UN Region,IMF Forecast,IMF Year
0,World,—,105568776,2023
1,United States,Americas,26854599,2023
2,China,Asia,19373586,[n 1]2023
3,Japan,Asia,4409738,2023
4,Germany,Europe,4308854,2023
5,India,Asia,3736882,2023
6,United Kingdom,Europe,3158938,2023
7,France,Europe,2923489,2023
8,Italy,Europe,2169745,2023
9,Canada,Americas,2089672,2023


Mengubah nilai dari teks menjadi angka, mengambil tahun dari string yang berantakan, dan mengonversi
GDP dari Juta ke Miliar USD

In [13]:
df1["IMF Forecast"] = pd.to_numeric(df1["IMF Forecast"], errors='coerce')
df1["IMF Year"] = df1["IMF Year"].astype(str).str.extract(r'(\d{4})').astype(int)
df1["IMF Forecast"] = df1["IMF Forecast"].astype("int")
df1["IMF Forecast"] = df1["IMF Forecast"]/1000
df1["IMF Forecast"] = np.round(df1["IMF Forecast"], 2)
df1.rename(columns={"IMF Forecast": "IMF Forecast (Billion USD)"}, inplace=True)
df1

,Country,UN Region,IMF Forecast (Billion USD),IMF Year
0,World,—,105568.78,2023
1,United States,Americas,26854.60,2023
2,China,Asia,19373.59,2023
3,Japan,Asia,4409.74,2023
4,Germany,Europe,4308.85,2023
5,India,Asia,3736.88,2023
6,United Kingdom,Europe,3158.94,2023
7,France,Europe,2923.49,2023
8,Italy,Europe,2169.74,2023
9,Canada,Americas,2089.67,2023


Merapikan seluruh tabel (sumber IMF, World Bank, UN) lalu menyimpan dua dataset sebagai CSV

In [14]:
df2 = df.copy()
df2.columns = ['Country', 'UN Region', 'IMF Forecast', 'IMF Year', 'Estimate (World Bank)', 'Year (World Bank)', 'Estimate (United Nation)', 'Year (United Nation)']


year_cols = ["IMF Year", "Year (World Bank)", "Year (United Nation)"]
for col in year_cols:
    df2[col] = df2[col].astype(str).str.extract(r'(\d{4})')[0].fillna(0).astype(int)

gdp_cols = ["IMF Forecast", "Estimate (World Bank)", "Estimate (United Nation)"]
df2[gdp_cols] = df2[gdp_cols].apply(pd.to_numeric, errors='coerce').fillna(0).astype(int)
df2[gdp_cols] = df2[gdp_cols] / 1000
df2[gdp_cols] = np.round(df2[gdp_cols], 2)

df2

,Country,UN Region,IMF Forecast,IMF Year,Estimate (World Bank),Year (World Bank),Estimate (United Nation),Year (United Nation)
0,World,—,105568.78,2023,100562.01,2022,96698.00,2021
1,United States,Americas,26854.60,2023,25462.70,2022,23315.08,2021
2,China,Asia,19373.59,2023,17963.17,2022,17734.13,2021
3,Japan,Asia,4409.74,2023,4231.14,2022,4940.88,2021
4,Germany,Europe,4308.85,2023,4072.19,2022,4259.94,2021
...,...,...,...,...,...,...,...,...
209,Anguilla,Americas,0.00,0,0.00,0,0.30,2021
210,Kiribati,Oceania,0.25,2023,0.22,2022,0.23,2021
211,Nauru,Oceania,0.15,2023,0.15,2022,0.16,2021
212,Montserrat,Americas,0.00,0,0.00,0,0.07,2021


Load DataFrame ke dalam file CSV bernama "Largest_economies.csv"

In [ ]:
# Load the DataFrame to the CSV file named "Largest_economies.csv"
path = r"C:\Users\ASUS\Downloads\Last Practice\Largest_economies.csv"
df1.to_csv(path, index=False)

In [ ]:
path = r"C:\Users\ASUS\Downloads\Last Practice\GDP_by_Country.csv"
df2.to_csv(path, index=False)